## 02 - Band SNR ratios and per-epoch table

Processes a recording only where outputs are missing (raw CSV, band SNR CSVs, epoch table), plots band SNR time courses (delta/theta/alpha/beta from `ecog.spectral.BAND_RATIOS`, with an example override) and builds the per-epoch table (aperiodic fit, band power, state metrics).

Writes `cache/<animal>/<band>/*.csv`, `cache/epochs_02/`, and under `outputs/02_band_ratios_and_epoch_table/` the band figure, `<label>_ch<c>_epoch_table.csv`, `<label>_ch<c>_fit_quality.csv` and `run_info.json`.

### Variables

In [ ]:
EDF_FOLDER = "../../Completed_Analyses/Emergence/raw_edfs"  # folder holding the .edf files (relative to this notebook)
EDF = "EmergRev_3Perc_Induction_PEEG02M_2026-08-27_15_49_57_export.edf"  # .edf file name inside EDF_FOLDER
ANIMAL_ID = "PEEG02M"  # animal id (label and cache sub-folder)
CONDITION = "Induction"  # condition name (label and raw CSV column)
RECORDING_NUMBER = 1  # session number (label)
CHANNEL = 0  # EDF channel analysed (0/1 = EEG)
EVENTS = {"onset": 1800}  # named events in recording-clock seconds; ONSET_EVENT and ALIGN_EVENT must be among them
BANDS = ["delta", "theta", "alpha", "beta", "low_delta"]  # band SNR series to compute: keys of ecog.spectral.BAND_RATIOS or of BAND_DEFINITIONS
# extra or overriding band SNR definitions: signal band vs flanking noise bands (Hz); a new name adds a band, an existing name replaces it
BAND_DEFINITIONS = {"low_delta": {"signal": (1.0, 2.5), "noise": [(0.5, 1.0), (2.5, 4.0)]}}
OVERWRITE_RAW = False  # True re-extracts the raw CSV
OVERWRITE_BANDS = False  # True recomputes the band CSVs; required after redefining an existing band name (cached CSVs are not checked against definitions)
OVERWRITE_EPOCHS = False  # True refits the epoch table even when the cache matches design and settings
ALIGN_EVENT = "onset"  # event placed at x = 0 in the band plot
BIN_S = 120  # time bin of the plotted band series (s)
SMOOTH_S = 60  # halflife (s) of the exponential smoothing of the plotted line; None = unsmoothed
TICK_STEP = 600  # x tick spacing in seconds
ONSET_EVENT = "onset"  # event at 0 on the epoch time axis
EPOCH_S = 30  # epoch length of the epoch table (s)
WELCH_WINDOW_S = 4  # Welch sub-window inside an epoch (s); frequency resolution = 1 / WELCH_WINDOW_S
FIT_RANGE = (1.0, 20.0)  # Hz range of the aperiodic fit, spectral entropy, median frequency and SEF95
EPOCH_BANDS = {"delta": (1.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0)}  # Hz bands summarised per epoch (<band>_abs, _rel, _periodic columns)
WORK_DIR = "cache"  # processing cache (raw, band and epoch tables), reused between runs
OUTPUT_DIR = "outputs/02_band_ratios_and_epoch_table"  # figures and tables; files are never overwritten
FIGURE_FORMATS = ("png", "svg")  # one file per format for each figure
DPI = 300  # resolution of the png

In [ ]:
from pathlib import Path  # path handling
import sys  # module search path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ecog").is_dir())  # repo root = first parent folder containing ecog/
sys.path.insert(0, str(ROOT))  # makes `import ecog` work without installing anything
import os  # path normalising
EDF_DIR = Path(os.path.abspath(EDF_FOLDER))  # absolute EDF folder with any `..` collapsed (relative paths start at this notebook's folder)
import mne  # EDF reader used by ecog
mne.set_log_level("WARNING")  # hides the EDF reader's progress messages
import contextlib, io, importlib  # output control and dynamic import
with contextlib.redirect_stderr(io.StringIO()):  # fooof prints a deprecation notice to stderr when it is first imported
    importlib.import_module("fooof")  # imported once here, before ecog needs it
import matplotlib.pyplot as plt  # figure creation
from ecog import (Recording, extractRecording, computeBandRatios, bandTable, buildEpochTable, fitQuality, BAND_RATIOS,
                  saveFigure, saveTable, writeRunInfo)  # processing, band tables, epoch table, saving
from ecog.plots import plotRatioSNR  # binned band time course
WORK = Path(WORK_DIR)  # cache folder
OUT = Path(OUTPUT_DIR)  # output folder

### 1. Describe the recording

In [ ]:
# everything the processing functions need to know about this recording
rec = Recording(animal_id=ANIMAL_ID, condition=CONDITION, recording_number=RECORDING_NUMBER, edf=EDF, edf_folder=EDF_DIR,
                channel_index=CHANNEL, events=EVENTS)
rec.validate()  # stops with one message if the file, channel or events are wrong
rec.label  # label used in every file name

### 2. Process only what is missing (existing CSVs are reused unless an OVERWRITE flag is True)

In [ ]:
DEFINITIONS = {**BAND_RATIOS, **BAND_DEFINITIONS}  # band SNR definitions in use: library defaults plus your additions/overrides
def onDisk():  # which outputs exist in the cache
    return {"raw": rec.rawCSV(WORK, CHANNEL).exists(), **{b: rec.bandCSV(WORK, CHANNEL, b).exists() for b in BANDS}}  # True = file present


print("before:", onDisk())  # status before processing
extractRecording(rec, WORK, overwrite=OVERWRITE_RAW)  # EDF -> raw CSV; skipped when present
computeBandRatios(rec, WORK, BANDS, definitions=DEFINITIONS, overwrite=OVERWRITE_BANDS)  # one SNR CSV per band; skipped per band when present
print("after: ", onDisk())  # status after processing

### 3. Band definitions in use (signal band over mean of the flanking noise bands, Hz)

In [ ]:
{b: DEFINITIONS[b] for b in BANDS}  # edit BANDS / BAND_DEFINITIONS above to change them

### 4. Band SNR time courses (one panel per band)

In [ ]:
fig, axes = plt.subplots(len(BANDS), 1, figsize=(5, 1.3 * len(BANDS)), sharex=True)  # one panel per band
for ax, band in zip(axes, BANDS):  # one band per panel
    series = bandTable(rec, CHANNEL, WORK, band, align_event=ALIGN_EVENT)  # band CSV in long form, time re-zeroed on ALIGN_EVENT
    plotRatioSNR(series, f"{band} SNR", ax=ax, binning_s=BIN_S, smoothing_s=SMOOTH_S, tick_step=TICK_STEP)  # binned mean +/- spread
    ax.axvline(0, color="black", ls="--", lw=1)  # ALIGN_EVENT
    ax.set_xlabel("")  # only the lowest panel keeps the axis label
    if ax.get_legend():  # single recording: no legend needed
        ax.get_legend().remove()  # drop it
axes[-1].set_xlabel("Time (minutes)")  # shared time axis
fig.tight_layout()  # spacing between panels
saveFigure(fig, OUT / f"{rec.label}_ch{CHANNEL}_band_snr", dpi=DPI, formats=FIGURE_FORMATS)  # saved without overwriting

### 5. Per-epoch table (cached in cache/epochs_02 until recording or settings change)

In [ ]:
# one row per EPOCH_S epoch: aperiodic exponent/offset, band abs/rel/periodic power, entropy, median frequency, suppression
table, spectra = buildEpochTable({CONDITION: [(rec, CHANNEL)]}, WORK, cache=WORK / "epochs_02", force=OVERWRITE_EPOCHS,
                                 onset_event=ONSET_EVENT, epoch_s=EPOCH_S, welch_window_s=WELCH_WINDOW_S,
                                 fit_range=FIT_RANGE, bands=EPOCH_BANDS)
print(len(table), "epochs of", EPOCH_S, "s")  # table size
table[["timeSeg", "exponent", "offset", "fit_r2", "delta_abs", "delta_rel", "delta_periodic", "spectral_entropy", "median_freq"]].head()  # first rows; timeSeg = epoch centre (s) relative to ONSET_EVENT

### 6. Column groups, fit quality of the aperiodic model, and save

In [ ]:
groups = ["recording", "condition", "animal_id", "mouse", "channel", "timeSeg"]  # identifying columns
for band in EPOCH_BANDS:  # columns generated for each band in EPOCH_BANDS
    print(band, [c for c in table.columns if c.startswith(band + "_")])  # abs, rel, periodic and fitted-peak columns
print([c for c in table.columns if c not in groups and c.split("_")[0] not in EPOCH_BANDS])  # background fit and state metrics
path = saveTable(table, OUT / f"{rec.label}_ch{CHANNEL}_epoch_table.csv")  # epoch table as csv, never overwritten
fit = fitQuality(table)  # fit R2 and error over every epoch of the recording (SD is NaN: one animal)
fitPath = saveTable(fit, OUT / f"{rec.label}_ch{CHANNEL}_fit_quality.csv")  # fit quality as csv, never overwritten
info = writeRunInfo(OUT, settings=dict(edf=EDF, channel=CHANNEL, bands=BANDS, band_definitions=BAND_DEFINITIONS, epoch_s=EPOCH_S,
                                       fit_range=FIT_RANGE, epoch_bands=EPOCH_BANDS, onset_event=ONSET_EVENT))  # versions and settings
print(path, fitPath, info)  # files written
fit  # fit quality of this recording